In [34]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score

In [2]:
df = pd.read_csv("data/heart.csv")
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


In [3]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns = ['target']), df['target'], test_size=0.3, random_state=0)

In [10]:
clf = LogisticRegression(max_iter=1000)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

# True Positive Rate (TPR) -> Benefit :
### Out of all actual positive how many we predicted Positive?
## `TPR = TP + (TP + FN)`
### TPR is also called the benefit of the model. 
Let's say you are making Email spam Classifier `1 -> Spam` and `0 -> Not Spam`.
Your Model predicted `80` Spam out of `100` actual Spam and your `TPR = 0.8` means you are able to take `80% benefit` of your model.

At best case you want your model to predict spam to every spam means `TPR = 1`, `100% benefit` of the model

-> You want your model to get TPR as maximum as possible .

In [39]:
cm = confusion_matrix(y_test, y_pred)
print(cm)
TN = cm[0,0] # True Negative
FP = cm[0,1] # False Positive
FN = cm[1,0] # False Negative
TP = cm[1,1] # True Positive

[[117  28]
 [ 13 150]]


In [40]:
TPR = TP / (TP + FN)
print(TPR)

0.9202453987730062


Means My Logistic Regression is only able identify 92 % of the actual spam.

I'm able to take 92% of the benefit 

---

# False Positive Rate (FPR) -> Cost :

### Out of all actual negative how many model is prediting Positive

## `FPR = FP / (FP + TN)`

### This is also called the `Cost` of the Model.

Let's take the same Email Spam Classifier. Your model is predicting Spam `20(FP)` of the `100(FP+TN)` Not Spam Email and So your `FPR = 0.2` means you are taking `20% Cost` which is not good because you may be flagging Spam to Important Non-Spam Emails.

At best case you want your model to predict Non-Spam to Non-Spam means `FPR = 0`, `0% Cost` of the model.

-> You want your model to get FPR as minimum as possible.

In [41]:
FPR = FP / (FP + TN)
print(FPR)

0.19310344827586207


Means My Logistic Regression is only cost me 19%

# Threshold :

As we know that Classification Models predict the probability of the data is positive. By default the threshold in the model is 0.5

For Binary Classification's default threshold:<br>
`probability >= 0.5 : 1`<br>
`probability < 0.5 : 0`<br>

we can change the threshold to 0.75, then<br>
`probability >= 0.75 : 1`<br>
`probability < 0.75 : 0`<br>

# ROC Curve

### ROC curve is a graph of `TPR vs. FPR` with different values of Threshold. It is used the find the best Thresholf value for the model.

In [24]:
y_pred_proba = clf.predict_proba(X_test)[:,1] # we want probability of positive

In [26]:
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
print(f"FPRs : \n {fpr}")
print(f"TPRs : \n {tpr}")
print(f"Thresholds : \n {thresholds}")

FPRs : 
 [0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.00689655 0.00689655 0.00689655 0.00689655
 0.0137931  0.0137931  0.0137931  0.0137931  0.0137931  0.0137931
 0.02068966 0.02068966 0.02068966 0.02068966 0.02758621 0.02758621
 0.02758621 0.02758621 0.02758621 0.03448276 0.03448276 0.03448276
 0.03448276 0.04827586 0.04827586 0.06206897 0.06206897 0.06206897
 0.07586207 0.07586207 0.07586207 0.07586207 0.08965517 0.08965517
 0.08965517 0.10344828 0.10344828 0.10344828 0.10344828 0.11034483
 0.11034483 0.11724138 0.11724138 0.11724138 0.13103448 0.13103448
 0.13103448 0.13103448 0.13793103 0.13793103 0.15172414 0.15172414
 0.15172414 0.17241379 0.17241379 0.17241379 0.17241379 0.17241379
 0.17931034 0.17931034 0.19310345 0.19310345 0.2        0.2137931
 0.2137931  0.22758621 0.22758621 0.24137931 0.24137931 0.24827586
 0.24827586 

In [38]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

trace0 = go.Scatter(
    x = fpr,
    y = tpr,
    mode = 'lines',
    name = 'ROC Curve'
) 

n = 5
indices = np.arange(len(thresholds)) % n == 0

trace1 = go.Scatter(
    x = fpr[indices],
    y = tpr[indices],
    mode = 'markers+text',
    name = 'Threshold Points',
    text = [f"{thr:.2f}" for thr in thresholds[indices]],
    textposition='top center'
)

trace2 = go.Scatter(
    x = [0,1],
    y = [0,1],
    mode = 'lines',
    name = 'Random (Are = 0.5)',
    line = dict(dash = 'dash')
)

data = [trace0, trace1, trace2]

layout = go.Layout(
    title = "ROC",
    xaxis = dict(title = 'FPR'),
    yaxis = dict(title = 'TPR'),
    autosize = False,
    width = 800,
    height = 800,
    showlegend = False
)

fig = go.Figure(data = data, layout=layout)

fig.show()

The best value of Threshold will be where the TPR is maximum and FPR is minimum that is 0.55 in the graph.

# Area Under Curve (AUC) :

### It is the area covered by the ROC curve. It provides overall performance of the model. It is used to compare two Classification Model. Model with higher AUC will be the best

In [ ]:

# SVM requires feature scaling for better performance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Logistic Regression model
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
lr_scores = lr_model.predict_proba(X_test)[:,1]

# SVM model
svm_model = SVC(probability=True)
svm_model.fit(X_train_scaled, y_train)
svm_scores = svm_model.predict_proba(X_test_scaled)[:,1]

# ROC curve data for logistic regression model
lr_fpr, lr_tpr, lr_thresholds = roc_curve(y_test, lr_scores)
lr_auc = roc_auc_score(y_test, lr_scores)

# ROC curve data for SVM model
svm_fpr, svm_tpr, svm_thresholds = roc_curve(y_test, svm_scores)
svm_auc = roc_auc_score(y_test, svm_scores)

# a trace for the Logistic Regression ROC curve
trace0 = go.Scatter(
    x=lr_fpr,
    y=lr_tpr,
    mode='lines',
    name=f'Logistic Regression (Area = {lr_auc:.2f})'
)

# a trace for the SVM ROC curve
trace1 = go.Scatter(
    x=svm_fpr,
    y=svm_tpr,
    mode='lines',
    name=f'SVM (Area = {svm_auc:.2f})'
)

# Diagonal line
trace2 = go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode='lines',
    name='Random (Area = 0.5)',
    line=dict(dash='dash')
)

data = [trace0, trace1, trace2]

# Define layout with square aspect ratio
layout = go.Layout(
    title='Receiver Operating Characteristic',
    xaxis=dict(title='False Positive Rate'),
    yaxis=dict(title='True Positive Rate'),
    autosize=False,
    width=800,
    height=800,
    showlegend=True
)

# Define figure and add data
fig = go.Figure(data=data, layout=layout)

# Show figure
fig.show()


As we can see that Support Vector Classifier has higher AUC that says the SVC will perform better than Logistic Regression